# Mushroom body: second-circuit generality test (Phase 7)

This notebook applies the **same pipeline** as the HD-ring validation to the mushroom body Kenyon cells + APL. The point is to show that `connectome -> subgraph -> parameterizer -> simulator` is genuine infrastructure: it works on a circuit with completely different topology than the HD ring.

| Axis | HD ring | Mushroom body |
|---|---|---|
| Topology | Recurrent ring | Feedforward + global inhibition |
| Size | 130 | 1924 (1923 KCs + 1 APL) |
| Target dynamics | Localized bump | Sparse k-WTA coding |
| Activation | tanh (relu blows up under recurrence) | relu (no recurrent excitation to amplify) |
| Symmetrize | True (asymmetry kills the bump) | False (breaks one-way inhibition) |

**What broke during Phase 7** and was fixed in the library:

1. `HemibrainConnectome.query(ids=...)` did not cache results. APL has no `type` field in hemibrain:v1.2.1, so we can only fetch it by id; without caching, fixture-only tests couldn't load it. Fixed by adding a `neurons.by_id.<hash>.parquet` cache, mirroring the adjacency-hash scheme.
2. APL also has no NT label. The mushroom-body loader patches its `cell_type` and `nt` post-query (`circuits.mushroom_body._patch_apl`). The library convention -- NT comes from the backend or a `nt_by_type` map -- doesn't cover singletons, so a per-circuit fixup is the right place.

**What didn't need to change:** parameterizer, simulator, ModelSpec, cache primitive, stimulus utilities. The infrastructure assumptions held.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from galvani import ParameterizerOptions, default_parameterizer, simulate
from galvani.circuits.mushroom_body import MB_NT, load_mushroom_body
from galvani.connectome.cache import ParquetCache
from galvani.connectome.hemibrain import HemibrainConnectome
from galvani.model.rate import relu
from galvani.model.spec import ModelSpec

FIXTURES = Path("..") / "tests" / "fixtures"
conn = HemibrainConnectome(cache=ParquetCache(FIXTURES), nt_by_type=MB_NT, token="fake")
mb = load_mushroom_body(conn)
apl = mb.subgraph.neurons[mb.apl_index]
print(f"loaded MB: N={len(mb.subgraph.neurons)}, KCs={mb.n_kc}, APL at index {mb.apl_index}")
print(f"  APL bodyId={apl.id}, cell_type={apl.cell_type!r}, nt={apl.nt!r}")
print(f"  synapse rows: {int(mb.subgraph.counts.size)}")
print(f"  total synapses: {int(mb.subgraph.counts.sum())}")

## APL's role in the weight matrix

After parameterization the column of `W` corresponding to APL should be entirely non-positive (APL is GABAergic) and quantitatively comparable to KC -> KC excitation.

In [ ]:
spec = default_parameterizer(mb.subgraph, ParameterizerOptions(global_gain=0.08))
apl_out = spec.global_gain * spec.weights[:, mb.apl_index]
apl_in = spec.global_gain * spec.weights[mb.apl_index]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(apl_out, bins=60, color="crimson", alpha=0.85)
ax1.set_xlabel("weight from APL -> KC (gain-scaled)")
ax1.set_ylabel("# of KCs")
ax1.set_title("APL outgoing weights (must be <=0)")
ax2.hist(apl_in, bins=60, color="royalblue", alpha=0.85)
ax2.set_xlabel("weight from KC -> APL (gain-scaled)")
ax2.set_ylabel("# of KCs")
ax2.set_title("APL incoming weights (mostly >=0)")
plt.tight_layout()
plt.show()

print(f"APL outgoing: min={apl_out.min():.3f}, max={apl_out.max():.3f}, mean={apl_out.mean():.3f}")
print(
    f"APL outgoing nonzero, all <= 0? {(apl_out[apl_out != 0] <= 0).all()}  "
    f"({int((apl_out < 0).sum())}/{int((apl_out != 0).sum())} negative)"
)

## Sparse coding via APL feedback

Drive 30% of KCs externally (a moderately broad "odor"). With APL intact, the network should land at 1-10% active KCs after the feedback loop settles. Sweeping `global_gain` traces the sparseness curve.

In [ ]:
rng = np.random.default_rng(7)
n_kc = mb.n_kc
kc_indices = np.where(mb.kc_mask)[0]
chosen = rng.choice(kc_indices, size=int(0.30 * n_kc), replace=False)
pat = np.zeros(spec.n_neurons, dtype=np.float64)
pat[chosen] = 0.5


def constant_pat(_t):
    return pat


gains = np.array([0.02, 0.03, 0.05, 0.08, 0.12, 0.2, 0.5, 1.0])
active_fracs = []
apl_acts = []
for gain in gains:
    s = default_parameterizer(mb.subgraph, ParameterizerOptions(global_gain=float(gain)))
    r = simulate(s, duration=0.2, stimulus=constant_pat, activation=relu, dt=5e-4)
    kc_final = r.rates[-1, mb.kc_mask]
    apl_acts.append(float(r.rates[-1, mb.apl_index]))
    active = int((kc_final > 0.01 * kc_final.max()).sum()) if kc_final.max() > 0 else 0
    active_fracs.append(active / n_kc)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.semilogx(gains, np.array(active_fracs) * 100, "o-", lw=1.2)
ax1.axhline(5, color="gray", lw=0.5, ls="--", label="~5% target")
ax1.axhspan(1, 10, color="green", alpha=0.1, label="sparse-coding regime")
ax1.set_xlabel("global gain")
ax1.set_ylabel("active KCs (%)")
ax1.set_title("Sparseness vs gain")
ax1.legend()

ax2.semilogx(gains, apl_acts, "o-", color="crimson", lw=1.2)
ax2.set_xlabel("global gain")
ax2.set_ylabel("APL steady-state rate")
ax2.set_title("APL recruitment")
plt.tight_layout()
plt.show()

## Counterfactual: APL ablation eliminates sparsity

Zeroing APL's outgoing weights removes the global-inhibition feedback. The same KC input drive now produces a much denser activation. This is the canonical APL functional signature in the literature (Lin et al. 2014, Honegger et al. 2011).

In [ ]:
def ablate_apl(spec_in: ModelSpec) -> ModelSpec:
    w = spec_in.weights.copy()
    w[:, mb.apl_index] = 0.0
    return ModelSpec(
        neuron_ids=spec_in.neuron_ids,
        weights=w,
        tau=spec_in.tau,
        bias=spec_in.bias,
        global_gain=spec_in.global_gain,
        dataset_version=spec_in.dataset_version,
        defaults_used=dict(spec_in.defaults_used),
        notes={"apl_ablated": True},
    )


gain = 0.08
spec_with = default_parameterizer(mb.subgraph, ParameterizerOptions(global_gain=gain))
spec_without = ablate_apl(spec_with)

r_with = simulate(spec_with, duration=0.2, stimulus=constant_pat, activation=relu, dt=5e-4)
r_without = simulate(spec_without, duration=0.2, stimulus=constant_pat, activation=relu, dt=5e-4)

kc_with = r_with.rates[-1, mb.kc_mask]
kc_without = r_without.rates[-1, mb.kc_mask]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
ax1.bar(np.arange(n_kc), np.sort(kc_with)[::-1], width=1.0, color="royalblue")
ax1.set_title(f"WITH APL  (n_active = {int((kc_with > 0.01 * max(kc_with.max(), 1e-9)).sum())})")
ax1.set_xlabel("KC rank (by activity)")
ax1.set_ylabel("final rate")

ax2.bar(np.arange(n_kc), np.sort(kc_without)[::-1], width=1.0, color="crimson")
ax2.set_title(
    f"WITHOUT APL (n_active = {int((kc_without > 0.01 * max(kc_without.max(), 1e-9)).sum())})"
)
ax2.set_xlabel("KC rank (by activity)")
plt.tight_layout()
plt.show()

## Conclusion

The pipeline runs on a circuit that is feedforward + sparse + ~15x larger than the HD ring. The only library change Phase 7 prompted was caching the ids-only neuron-query path (so APL, which lacks a `type` field in hemibrain, can be served from committed fixtures). The parameterizer, simulator, ModelSpec, and stimulus utilities were unchanged. Sparse k-WTA dynamics emerge from connectome data without per-circuit weight tuning.

The pipeline is now infrastructure rather than HD-ring-specific code.